# In Class Activity 7

# TensorFlow Environment Check

First, verify that your **current Python/Jupyter environment** can run TensorFlow. Then continue into the deep-learning activity below.

Run the cells from top to bottom. Most students only need the information printed by these checks if something goes wrong.

**Important:** This notebook does not change your environment. It only checks what is currently installed.

## 1. What Python environment is this notebook using?

A very common problem is installing TensorFlow into one environment and running Jupyter from another. This cell shows exactly which Python executable the notebook is using.

In [ ]:
import sys
import platform
import os

print('Python executable:')
print(sys.executable)
print('\nPython version:')
print(sys.version)
print('\nOperating system:')
print(platform.platform())
print('\nCONDA_DEFAULT_ENV:')
print(os.environ.get('CONDA_DEFAULT_ENV', '(not a Conda environment)'))


### What should you look for?

The Python executable should belong to the environment you expect to use for this course, and the operating system should match the computer you are using.

If the path points to an unexpected environment, **restart Jupyter using the intended environment** or select the correct kernel in Jupyter.

## 2. Can Python import TensorFlow?

This is the basic TensorFlow test.

In [ ]:
try:
    import tensorflow as tf
    print('SUCCESS: TensorFlow imported.')
    print('TensorFlow version:', tf.__version__)
except Exception as e:
    print('FAILED: TensorFlow could not be imported.')
    print('\nError type:', type(e).__name__)
    print('Error message:', e)
    print('\nSee the troubleshooting section near the end of this notebook.')

## 3. Can TensorFlow perform a real calculation?

Importing TensorFlow is not enough. This cell performs a matrix multiplication using TensorFlow.

In [ ]:
import tensorflow as tf

try:
    a = tf.constant([[1.0, 2.0], [3.0, 4.0]])
    b = tf.constant([[5.0, 6.0], [7.0, 8.0]])
    c = tf.matmul(a, b)
    print(c.numpy())
    print('\nSUCCESS: TensorFlow completed a calculation.')
except Exception as e:
    print('FAILED during TensorFlow calculation.')
    print(type(e).__name__, ':', e)

## 4. Can Keras train a tiny neural network?

This is the most useful end-to-end test. It creates a very small binary classification problem and trains a neural network for a few epochs.

If this cell runs successfully, your environment is ready for the basic neural-network work in this course.

In [ ]:
import numpy as np
import tensorflow as tf

np.random.seed(42)
tf.random.set_seed(42)

# Simple synthetic binary classification data
X = np.random.randn(200, 2).astype('float32')
y = (X[:, 0] + X[:, 1] > 0).astype('float32')

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(2,)),
    tf.keras.layers.Dense(4, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history = model.fit(X, y, epochs=5, batch_size=32, verbose=1)

loss, accuracy = model.evaluate(X, y, verbose=0)
print(f'\nFinal training loss: {loss:.4f}')
print(f'Final training accuracy: {accuracy:.3f}')
print('\nSUCCESS: Keras trained a neural network.')

## 5. A compact diagnostic summary

Run this cell last. It gives you the information that is most useful to send to an instructor or use when troubleshooting.

In [ ]:
print('===== TensorFlow Diagnostic Summary =====')
print('Python:', sys.version.split()[0])
print('Python executable:', sys.executable)
print('Operating system:', platform.platform())
print('Conda environment:', os.environ.get('CONDA_DEFAULT_ENV', '(not detected)'))

try:
    print('TensorFlow:', tf.__version__)
    print('TensorFlow import: PASS')
except Exception as e:
    print('TensorFlow import: FAIL')
    print('Error:', type(e).__name__, e)
print('==========================================')


# Troubleshooting common problems

**About the command line:** Use the command-line program available on your computer. The commands below are written to work with Conda on both Windows and macOS.

**Using your own environment name:** Replace `YOUR_ENVIRONMENT_NAME` in an example with the actual name of your Conda environment (for example, `geol599` or `ml-course`).

## Problem 1: `ModuleNotFoundError: No module named 'tensorflow'`

TensorFlow is not installed in the Python environment used by this notebook.

At the **command line**, activate the environment you want to use and install TensorFlow:

```text
conda activate YOUR_ENVIRONMENT_NAME
python -m pip install tensorflow
```

Then restart Jupyter and select the kernel associated with your environment.

---

## Problem 2: TensorFlow is installed, but this notebook cannot find it

Compare the Python path printed in Section 1 with the environment in which you installed TensorFlow.

A reliable command-line check is:

```text
conda activate YOUR_ENVIRONMENT_NAME
python -c "import sys; print(sys.executable)"
python -c "import tensorflow as tf; print(tf.__version__)"
```

If that works at the command line but not in Jupyter, the notebook is probably using a different kernel.

---

## Problem 3: Jupyter does not show the TensorFlow environment

Install/register the kernel from the command line:

```text
conda activate YOUR_ENVIRONMENT_NAME
python -m pip install ipykernel
python -m ipykernel install --user --name YOUR_ENVIRONMENT_NAME --display-name "Python (YOUR_ENVIRONMENT_NAME)"
```

Then restart Jupyter.

---

## Problem 4: Package/version conflicts

Avoid repeatedly installing random TensorFlow-related packages into `base`. A clean environment is usually easier to diagnose.

For this course, a clean environment can be created with:

```text
conda create -n YOUR_ENVIRONMENT_NAME python=3.11 -y
conda activate YOUR_ENVIRONMENT_NAME
python -m pip install tensorflow numpy pandas matplotlib scikit-learn seaborn jupyterlab ipykernel
python -m ipykernel install --user --name YOUR_ENVIRONMENT_NAME --display-name "Python (YOUR_ENVIRONMENT_NAME)"
```

---

## Final check

You are ready to proceed when:

- TensorFlow imports successfully.
- The matrix multiplication test succeeds.
- The tiny Keras model trains without an exception.



# Deep Learning Activity: Visualizing Deep Learning Concepts

We will use two Landsat spectral bands to distinguish **cotton crop** from **soil with vegetation stubble**. The companion workbook follows the same sequence.

**Learning goals:** explain activation functions, compare logistic regression with a neural network, use validation to recognize underfitting and overfitting, and explain what epochs, dropout, early stopping, hidden units, and regularization are doing.

After completing the environment check above, this workbook uses TensorFlow/Keras for the neural-network sections. Logistic regression remains the linear baseline. The goal is to connect the model code directly to the deep-learning concepts being visualized.


In [ ]:
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 7
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)
sns.set_theme(style='whitegrid')
print('TensorFlow version:', tf.__version__)


## 1. Load an earth-observation dataset

The Statlog Landsat Satellite data contain multispectral measurements for land-cover pixels. We intentionally use only two central-pixel bands here so that we can visualize a decision boundary.


In [ ]:
url = 'https://raw.githubusercontent.com/dblaskey/ML_Course_Code/main/Data/Landsat_Landcover.csv'

landsat = (
    pd.read_csv(url)
    # In this CSV, 2 = cotton crop and 5 = soil with vegetation stubble.
    .query('land_cover in [2, 5]')
    .assign(land_cover=lambda data: np.where(
        data['land_cover'].eq(5), 'soil with vegetation stubble', 'cotton crop'
    ))
    .dropna(subset=['pixel_5_band_1', 'pixel_5_band_2'])
    .loc[:, ['land_cover', 'pixel_5_band_1', 'pixel_5_band_2']]
    .rename(columns={'pixel_5_band_1': 'band_1', 'pixel_5_band_2': 'band_2'})
)

display(landsat['land_cover'].value_counts())

sns.scatterplot(data=landsat, x='band_1', y='band_2', hue='land_cover', alpha=0.65, s=24)
plt.title('Two spectral bands from a Landsat pixel')
plt.show()


## 2. Split and scale

Neural networks are trained with gradient-based optimization, so predictors should be on comparable scales. Notice that the scaler is fit to the training data only.


In [ ]:
X = landsat[['band_1', 'band_2']]

# Model target: 1 = soil with vegetation stubble, 0 = cotton crop.
# Keeping the target numeric makes scikit-learn's early-stopping path more stable.
y = (landsat['land_cover'] == 'soil with vegetation stubble').astype(int)

class_names = ['cotton crop', 'soil with vegetation stubble']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
scaled_train = pd.DataFrame(X_train_scaled, columns=['band_1', 'band_2'])
scaled_train['land_cover'] = y_train.map({0: 'cotton crop', 1: 'soil with vegetation stubble'}).to_numpy()
scaled_train['stage'] = 'After scaling'

raw_train = X_train.copy()
raw_train['land_cover'] = y_train.map({0: 'cotton crop', 1: 'soil with vegetation stubble'}).to_numpy()
raw_train['stage'] = 'Before scaling'

scaling_plot_data = pd.concat([raw_train, scaled_train], ignore_index=True)

g = sns.relplot(
    data=scaling_plot_data,
    x='band_1',
    y='band_2',
    hue='land_cover',
    col='stage',
    kind='scatter',
    alpha=0.55,
    s=20,
    facet_kws={'sharex': False, 'sharey': False},
)
g.fig.suptitle('Feature scaling changes the coordinate system, not the class labels', y=1.05)
plt.show()


## 3. Logistic-regression baseline

Logistic regression can learn one linear boundary in the two-feature space. This is our baseline, not an opponent to be defeated at all costs.


In [ ]:
logistic_model = LogisticRegression(random_state=RANDOM_STATE)
logistic_model.fit(X_train_scaled, y_train)

logistic_pred = logistic_model.predict(X_test_scaled)
print(f'Logistic-regression test accuracy: {accuracy_score(y_test, logistic_pred):.3f}')


## 4. Build a small neural network

This model has one hidden layer with eight sigmoid units. Each hidden unit can create a different bend in the decision surface; together they can represent more complex boundaries. We use TensorFlow/Keras so the model directly reflects the neural-network structure used in deep learning.


In [ ]:
def build_network(hidden_units=8, decay=0.001, max_epochs=500, early_stopping=False,
                  dropout_rate=0.0, validation_data=None):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(2,)),
        tf.keras.layers.Dense(
            hidden_units,
            activation='sigmoid',
            kernel_regularizer=tf.keras.regularizers.l2(decay) if decay > 0 else None,
        ),
    ])

    if dropout_rate > 0:
        model.add(tf.keras.layers.Dropout(dropout_rate))

    model.add(tf.keras.layers.Dense(1, activation='sigmoid'))
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
        loss='binary_crossentropy',
        metrics=['accuracy'],
    )

    callbacks = []
    if early_stopping and validation_data is not None:
        callbacks.append(tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=10,
            min_delta=1e-3,
            restore_best_weights=True,
        ))

    return model, callbacks


def get_stubble_probability(model, new_data):
    return model.predict(new_data, verbose=0).ravel()


nn_model, nn_callbacks = build_network(hidden_units=8, decay=0.001)
nn_history = nn_model.fit(
    X_train_scaled, y_train,
    epochs=500,
    batch_size=32,
    verbose=0,
    callbacks=nn_callbacks,
)
print('Neural network trained.')


## Visualize the network architecture

This network has two input features, one hidden layer, and one output probability. A hidden unit is not a physical thing in the data; it is a learned transformation of the input features.


In [ ]:
input_nodes = pd.DataFrame({
    'layer': 'Input',
    'node': ['band_1', 'band_2'],
    'x': [1, 1],
    'y': [2.5, 1.5],
})
hidden_nodes = pd.DataFrame({
    'layer': 'Hidden layer',
    'node': [f'h{i}' for i in range(1, 9)],
    'x': 2,
    'y': np.linspace(4, 0, 8),
})
output_nodes = pd.DataFrame({
    'layer': 'Output',
    'node': ['P(stubble)'],
    'x': [3],
    'y': [2],
})
network_nodes = pd.concat([input_nodes, hidden_nodes, output_nodes], ignore_index=True)

plt.figure(figsize=(7, 4.5))
for _, a in input_nodes.iterrows():
    for _, b in hidden_nodes.iterrows():
        plt.plot([a.x, b.x], [a.y, b.y], color='gray', alpha=0.15)
for _, a in hidden_nodes.iterrows():
    for _, b in output_nodes.iterrows():
        plt.plot([a.x, b.x], [a.y, b.y], color='gray', alpha=0.15)

palette = {'Input': '#F2F2F2', 'Hidden layer': '#9ECAE1', 'Output': '#FDAE6B'}
for layer, data in network_nodes.groupby('layer'):
    plt.scatter(data.x, data.y, s=700, color=palette[layer], edgecolor='black', zorder=3, label=layer)
    for _, row in data.iterrows():
        plt.text(row.x, row.y, row.node, ha='center', va='center', fontsize=9, zorder=4)

plt.xticks([1, 2, 3], ['Input', 'Hidden', 'Output'])
plt.yticks([])
plt.title('A one-hidden-layer neural network')
plt.legend(loc='upper right')
plt.box(False)
plt.show()


## Visualize gradient descent and learning rate

Training repeatedly changes the weights to reduce loss. This simple example uses one fake parameter and a U-shaped loss curve so we can see the idea.


In [ ]:
def simulate_descent(learning_rate, start=-3, steps=12):
    parameter = start
    rows = []
    for step in range(steps + 1):
        loss = (parameter - 1.5) ** 2
        rows.append({'step': step, 'parameter': parameter, 'loss': loss, 'learning_rate': f'learning rate = {learning_rate}'})
        gradient = 2 * (parameter - 1.5)
        parameter = parameter - learning_rate * gradient
    return pd.DataFrame(rows)

loss_curve = pd.DataFrame({'parameter': np.linspace(-4, 5, 500)})
loss_curve['loss'] = (loss_curve['parameter'] - 1.5) ** 2

descent_paths = pd.concat([
    simulate_descent(0.05),
    simulate_descent(0.25),
    simulate_descent(0.9),
], ignore_index=True)

plt.figure(figsize=(8, 5))
plt.plot(loss_curve['parameter'], loss_curve['loss'], color='gray', linewidth=2)
sns.lineplot(data=descent_paths, x='parameter', y='loss', hue='learning_rate', marker='o')
plt.title('Learning rate controls how large each weight update is')
plt.xlabel('Parameter value')
plt.ylabel('Loss')
plt.show()


## 5. Use validation to choose model complexity

For this implementation, we compare a small range of hidden-layer sizes on a validation set. The test set remains untouched. Very few units may underfit; many units can fit noise and hurt validation performance.


In [ ]:
# Split the training data again to create a validation set.
# The test set remains untouched until final evaluation.

X_fit, X_validation, y_fit, y_validation = train_test_split(
    X_train_scaled,
    y_train,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y_train
)

print("Fit observations:", len(X_fit))
print("Validation observations:", len(X_validation))
print("Test observations:", len(X_test_scaled))

In [ ]:
validation_rows = []

for hidden_units in [1, 2, 4, 8, 16]:
    model, callbacks = build_network(hidden_units=hidden_units, decay=0.001)
    model.fit(X_fit, y_fit, epochs=200, batch_size=32, verbose=0, callbacks=callbacks)

    train_probability = get_stubble_probability(model, X_fit)
    validation_probability = get_stubble_probability(model, X_validation)
    validation_rows.append({
        'hidden_units': hidden_units,
        'training_accuracy': accuracy_score(y_fit, (train_probability >= 0.5).astype(int)),
        'validation_accuracy': accuracy_score(y_validation, (validation_probability >= 0.5).astype(int)),
    })

validation_results = pd.DataFrame(validation_rows)
validation_long = validation_results.melt(
    id_vars='hidden_units',
    var_name='data_split',
    value_name='accuracy',
)

sns.lineplot(
    data=validation_long,
    x='hidden_units',
    y='accuracy',
    hue='data_split',
    marker='o',
)
plt.xticks([1, 2, 4, 8, 16])
plt.title('Model complexity: training versus validation accuracy')
plt.xlabel('Hidden units')
plt.ylabel('Accuracy')
plt.show()

validation_results


## 6. What is an epoch?

An **epoch** means the model has made one full pass through the training data. In Keras, `epochs` specifies the maximum number of passes through the training data.

More epochs can reduce error up to a point, but more training is not automatically better. Eventually, the model may start fitting patterns that do not generalize.


In [ ]:
epoch_rows = []

for max_epochs in [10, 25, 50, 100, 200, 400]:
    model, callbacks = build_network(hidden_units=8, decay=0.001)
    history = model.fit(
        X_fit, y_fit,
        epochs=max_epochs,
        batch_size=32,
        verbose=0,
        callbacks=callbacks,
    )

    train_probability = get_stubble_probability(model, X_fit)
    validation_probability = get_stubble_probability(model, X_validation)
    epoch_rows.append({
        'max_epochs': max_epochs,
        'training_accuracy': accuracy_score(y_fit, (train_probability >= 0.5).astype(int)),
        'validation_accuracy': accuracy_score(y_validation, (validation_probability >= 0.5).astype(int)),
        'final_training_loss': history.history['loss'][-1],
        'epochs_used': len(history.history['loss']),
    })

epoch_results = pd.DataFrame(epoch_rows)
epoch_long = epoch_results.melt(
    id_vars=['max_epochs', 'final_training_loss', 'epochs_used'],
    value_vars=['training_accuracy', 'validation_accuracy'],
    var_name='data_split',
    value_name='accuracy',
)

sns.lineplot(
    data=epoch_long,
    x='max_epochs',
    y='accuracy',
    hue='data_split',
    marker='o',
)
plt.title('Epoch idea: more training is not always better')
plt.xlabel('Maximum epochs')
plt.ylabel('Accuracy')
plt.show()

epoch_results


## 7. Early stopping

**Early stopping** monitors validation performance during training and stops when validation performance stops improving. The goal is to keep the model from overfitting the training data.

Here Keras does this with the `EarlyStopping` callback. We monitor validation loss and restore the weights from the best epoch.


In [ ]:
early_stop_model, early_stop_callbacks = build_network(
    hidden_units=8,
    decay=0.001,
    early_stopping=True,
    validation_data=(X_validation, y_validation),
)
early_stop_history = early_stop_model.fit(
    X_fit, y_fit,
    epochs=500,
    batch_size=32,
    verbose=0,
    validation_data=(X_validation, y_validation),
    callbacks=early_stop_callbacks,
)

validation_losses = np.array(early_stop_history.history['val_loss'])
best_epoch = int(np.argmin(validation_losses))

print('Epochs used:', len(early_stop_history.history['loss']))
print('Best validation loss:', round(validation_losses[best_epoch], 3))
print('Best validation epoch:', best_epoch + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(early_stop_history.history['loss'])
axes[0].axvline(best_epoch, color='#C94D4D', linestyle='--', label='best validation epoch')
axes[0].set(
    title='Training loss by epoch',
    xlabel='Epoch',
    ylabel='Loss',
)
axes[0].legend()

axes[1].plot(early_stop_history.history['val_accuracy'])
axes[1].axvline(best_epoch, color='#C94D4D', linestyle='--', label='best validation epoch')
axes[1].set(
    title='Validation accuracy by epoch',
    xlabel='Epoch',
    ylabel='Validation accuracy',
)
axes[1].legend()

plt.tight_layout()
plt.show()


## 8. What is dropout?

**Dropout** is a regularization method. During training, the model randomly turns off some hidden units on each update. This makes the network less dependent on any single hidden unit and can reduce overfitting.

TensorFlow/Keras implements dropout directly with `tf.keras.layers.Dropout`. We will visualize the same idea and also fit a small network with dropout below.


In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
mask = rng.choice(
    ['kept', 'dropped'],
    size=(8, 10),
    p=[0.7, 0.3],
)

dropout_demo = pd.DataFrame(
    mask,
    index=[f'unit_{i}' for i in range(1, 9)],
    columns=[f'step_{i}' for i in range(1, 11)],
)

dropout_numeric = dropout_demo.eq('kept').astype(int)

plt.figure(figsize=(8, 4))
sns.heatmap(
    dropout_numeric,
    cmap=sns.color_palette(['lightgray', '#3B6FB6'], as_cmap=True),
    cbar=False,
    linewidths=0.5,
    linecolor='white',
)
plt.title('Dropout idea: random hidden units are ignored during training')
plt.xlabel('Training step')
plt.ylabel('Hidden unit')
plt.show()


In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
dropout_state = network_nodes.copy()
drop_mask = rng.choice([True, False], size=len(dropout_state), p=[0.7, 0.3])
dropout_state['state'] = np.where(
    dropout_state['layer'].eq('Hidden layer'),
    np.where(drop_mask, 'kept', 'dropped'),
    'always active',
)

plt.figure(figsize=(7, 4.5))
for _, a in input_nodes.iterrows():
    for _, b in hidden_nodes.iterrows():
        plt.plot([a.x, b.x], [a.y, b.y], color='gray', alpha=0.10)
for _, a in hidden_nodes.iterrows():
    for _, b in output_nodes.iterrows():
        plt.plot([a.x, b.x], [a.y, b.y], color='gray', alpha=0.10)

state_palette = {'always active': 'white', 'kept': '#3B6FB6', 'dropped': 'lightgray'}
for state, data in dropout_state.groupby('state'):
    plt.scatter(data.x, data.y, s=700, color=state_palette[state], edgecolor='black', zorder=3, label=state)
    for _, row in data.iterrows():
        plt.text(row.x, row.y, row.node, ha='center', va='center', fontsize=9, zorder=4)

plt.xticks([1, 2, 3], ['Input', 'Hidden', 'Output'])
plt.yticks([])
plt.title('Dropout as a network diagram: some hidden units are ignored')
plt.legend(loc='upper right')
plt.box(False)
plt.show()


## 9. Evaluate once on the test set


In [ ]:
nn_probability = get_stubble_probability(nn_model, X_test_scaled)
nn_pred = (nn_probability >= 0.5).astype(int)

print(f'Neural-network test accuracy: {accuracy_score(y_test, nn_pred):.3f}')
print(f'Neural-network test log loss: {log_loss(y_test, nn_probability):.3f}')

ConfusionMatrixDisplay.from_predictions(
    y_test,
    nn_pred,
    display_labels=class_names,
    cmap='Blues',
    values_format='d',
)
plt.title('Neural network: test set')
plt.show()


## 10. Visualize the neural-network decision surface

The background shows predicted probability across a feature grid. The dots are only the held-out test observations.


In [ ]:
def plot_decision_surface(model, X_test_raw, y_test, scaler):
    x_min, x_max = landsat['band_1'].min(), landsat['band_1'].max()
    y_min, y_max = landsat['band_2'].min(), landsat['band_2'].max()
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 250),
        np.linspace(y_min, y_max, 250),
    )
    grid_raw = pd.DataFrame({
        'band_1': xx.ravel(),
        'band_2': yy.ravel(),
    })
    grid_scaled = scaler.transform(grid_raw)
    probability = get_stubble_probability(model, grid_scaled).reshape(xx.shape)

    plt.figure(figsize=(7, 6))
    surface = plt.contourf(xx, yy, probability, levels=20, cmap='coolwarm', alpha=0.45)
    plt.contour(xx, yy, probability, levels=[0.5], colors='black', linewidths=1.2)

    scatter = plt.scatter(
        X_test_raw['band_1'],
        X_test_raw['band_2'],
        c=y_test,
        cmap='coolwarm',
        edgecolor='black',
        s=28,
    )

    plt.colorbar(surface, label='Predicted P(soil with vegetation stubble)')
    handles, _ = scatter.legend_elements()
    plt.legend(handles, class_names, title='Observed test class')
    plt.xlabel('Central-pixel band 1')
    plt.ylabel('Central-pixel band 2')
    plt.title('Neural-network decision surface: test data overlaid')
    plt.show()


plot_decision_surface(nn_model, X_test, y_test, scaler)


The probability surface is the smooth background. The decision boundary is only the contour where the model crosses 0.5. The probability surface tells us about confidence; the boundary tells us where the predicted class changes.


In [ ]:
def make_prediction_grid(n=250):
    x_min, x_max = landsat['band_1'].min(), landsat['band_1'].max()
    y_min, y_max = landsat['band_2'].min(), landsat['band_2'].max()
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, n),
        np.linspace(y_min, y_max, n),
    )
    grid_raw = pd.DataFrame({
        'band_1': xx.ravel(),
        'band_2': yy.ravel(),
    })
    grid_scaled = scaler.transform(grid_raw)
    return xx, yy, grid_raw, grid_scaled


xx, yy, grid_raw, grid_scaled = make_prediction_grid()

logistic_probability = logistic_model.predict_proba(grid_scaled)[:, 1].reshape(xx.shape)
neural_probability = get_stubble_probability(nn_model, grid_scaled).reshape(xx.shape)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True, constrained_layout=True)

for ax, title, probability in [
    (axes[0], 'Logistic regression', logistic_probability),
    (axes[1], 'Neural network', neural_probability),
]:
    surface = ax.contourf(xx, yy, probability, levels=20, cmap='coolwarm', alpha=0.45)
    ax.contour(xx, yy, probability, levels=[0.5], colors='black', linewidths=1.2)
    ax.scatter(X_test['band_1'], X_test['band_2'], c=y_test, cmap='coolwarm', edgecolor='black', s=18, alpha=0.8)
    ax.set_title(title)
    ax.set_xlabel('Central-pixel band 1')
    ax.set_ylabel('Central-pixel band 2')

fig.colorbar(surface, ax=axes, label='Predicted P(stubble)', shrink=0.85)
fig.suptitle('A linear boundary versus a learned nonlinear boundary')
plt.show()


In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

hidden_layer = nn_model.layers[0]
hidden_weights, hidden_bias = hidden_layer.get_weights()
hidden_activation = sigmoid(grid_scaled @ hidden_weights + hidden_bias)

units_to_plot = list(range(min(4, hidden_activation.shape[1])))
fig, axes = plt.subplots(2, 2, figsize=(10, 8), sharex=True, sharey=True, constrained_layout=True)
axes = axes.ravel()

for ax, unit in zip(axes, units_to_plot):
    activation_surface = hidden_activation[:, unit].reshape(xx.shape)
    surface = ax.contourf(xx, yy, activation_surface, levels=20, cmap='viridis')
    ax.scatter(X_test['band_1'], X_test['band_2'], color='black', s=6, alpha=0.25)
    ax.set_title(f'hidden unit {unit + 1}')
    ax.set_xlabel('Central-pixel band 1')
    ax.set_ylabel('Central-pixel band 2')

fig.colorbar(surface, ax=axes, label='Activation', shrink=0.85)
fig.suptitle('Hidden-unit activation maps')
plt.show()


## 11. Experiment together

Now we will make the experiments explicit. Each row below changes **one setting at a time** relative to the baseline neural network.


In [ ]:
def evaluate_experiment(label, hidden_units=8, decay=0.001, max_epochs=200):
    model, callbacks = build_network(
        hidden_units=hidden_units,
        decay=decay,
        max_epochs=max_epochs,
    )

    history = model.fit(
        X_fit, y_fit,
        epochs=max_epochs,
        batch_size=32,
        verbose=0,
        callbacks=callbacks,
    )

    validation_probability = get_stubble_probability(model, X_validation)
    validation_pred = (validation_probability >= 0.5).astype(int)

    test_probability = get_stubble_probability(model, X_test_scaled)
    test_pred = (test_probability >= 0.5).astype(int)

    return {
        'experiment': label,
        'hidden_units': hidden_units,
        'decay': decay,
        'max_epochs': max_epochs,
        'epochs_used': len(history.history['loss']),
        'validation_accuracy': accuracy_score(y_validation, validation_pred),
        'validation_log_loss': log_loss(y_validation, validation_probability),
        'test_accuracy': accuracy_score(y_test, test_pred),
        'test_log_loss': log_loss(y_test, test_probability),
    }


experiment_settings = [
    {'label': 'baseline: 8 hidden units', 'hidden_units': 8, 'decay': 0.001, 'max_epochs': 200},
    {'label': 'underfit: 1 hidden unit', 'hidden_units': 1, 'decay': 0.001, 'max_epochs': 200},
    {'label': 'larger: 32 hidden units', 'hidden_units': 32, 'decay': 0.001, 'max_epochs': 200},
    {'label': 'regularized: decay = 0.1', 'hidden_units': 8, 'decay': 0.1, 'max_epochs': 200},
    {'label': 'short training: 50 epochs', 'hidden_units': 8, 'decay': 0.001, 'max_epochs': 50},
    {'label': 'long training: 400 epochs', 'hidden_units': 8, 'decay': 0.001, 'max_epochs': 400},
]

experiment_results = pd.DataFrame([
    evaluate_experiment(**settings)
    for settings in experiment_settings
])

experiment_results


In [ ]:
def fit_experiment_model(hidden_units=8, decay=0.001, max_epochs=200):
    model, callbacks = build_network(
        hidden_units=hidden_units,
        decay=decay,
        max_epochs=max_epochs,
    )

    model.fit(
        X_fit, y_fit,
        epochs=max_epochs,
        batch_size=32,
        verbose=0,
        callbacks=callbacks,
    )
    return model


def plot_experiment_boundaries(experiment_settings):
    x_min, x_max = landsat['band_1'].min(), landsat['band_1'].max()
    y_min, y_max = landsat['band_2'].min(), landsat['band_2'].max()
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 250),
        np.linspace(y_min, y_max, 250),
    )
    grid_raw = pd.DataFrame({
        'band_1': xx.ravel(),
        'band_2': yy.ravel(),
    })
    grid_scaled = scaler.transform(grid_raw)

    fig, axes = plt.subplots(2, 2, figsize=(11, 9), sharex=True, sharey=True, constrained_layout=True)
    axes = axes.ravel()

    for ax, settings in zip(axes, experiment_settings):
        model = fit_experiment_model(
            hidden_units=settings['hidden_units'],
            decay=settings['decay'],
            max_epochs=settings['max_epochs'],
        )
        probability = get_stubble_probability(model, grid_scaled).reshape(xx.shape)

        surface = ax.contourf(xx, yy, probability, levels=20, cmap='coolwarm', alpha=0.45)
        ax.contour(xx, yy, probability, levels=[0.5], colors='black', linewidths=1.1)
        ax.scatter(
            X_test['band_1'],
            X_test['band_2'],
            c=y_test,
            cmap='coolwarm',
            edgecolor='black',
            s=18,
            alpha=0.8,
        )
        ax.set_title(settings['label'])
        ax.set_xlabel('Central-pixel band 1')
        ax.set_ylabel('Central-pixel band 2')

    fig.colorbar(surface, ax=axes, label='Predicted P(soil with vegetation stubble)', shrink=0.85)
    fig.suptitle('How model settings change the decision boundary')
    plt.show()

boundary_settings = experiment_settings[:4]
plot_experiment_boundaries(boundary_settings)


## Exit questions

1. Why does a network need an activation function between dense layers?
2. What is an epoch, conceptually?
3. What does early stopping try to prevent?
4. Why can dropout reduce overfitting?
5. What is a hidden unit doing here that logistic regression cannot do?